# Local Neighborhood — Scientific Validation

Purpose (spec `Idea.md` §33): inspect the source catalog, verify the RA/Dec/distance
→ Galactic XYZ coordinate transform, inspect the XYZ distribution of the initial
object set, and compare it against the Gould Belt / Radcliffe Wave / Local Bubble
scientific model layers — all as a validation step before the scene is handed to
the web renderer. This notebook is a research/validation tool (spec §33), not the
primary deliverable; it is not required to be re-run for every catalog change, only
whenever the pipeline or catalog changes enough to warrant re-checking it by eye.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from local_galactic_structures.catalog import load_catalog
from local_galactic_structures.gould_belt import load_gould_belt_model
from local_galactic_structures.radcliffe_wave import load_radcliffe_wave
from local_galactic_structures.local_bubble import load_local_bubble_model

objects = load_catalog(REPO_ROOT / "data" / "normalized" / "catalog.parquet")
gould_belt = load_gould_belt_model(REPO_ROOT / "models" / "gould_belt.yaml")
radcliffe_wave = load_radcliffe_wave(REPO_ROOT / "models" / "radcliffe_wave.csv")
local_bubble = load_local_bubble_model(REPO_ROOT / "models" / "local_bubble.yaml")

print(f"{len(objects)} catalog objects, Gould Belt + Radcliffe Wave + Local Bubble models loaded")

20 catalog objects, Gould Belt + Radcliffe Wave + Local Bubble models loaded


## 1. Inspect the source catalog

In [2]:
df = pd.DataFrame([
    {
        "id": o.id,
        "name": o.name,
        "object_type": o.object_type,
        "distance_pc": o.distance.value_pc,
        "distance_error_pc": o.distance.error_pc,
        "x_pc": o.cartesian.x_pc,
        "y_pc": o.cartesian.y_pc,
        "z_pc": o.cartesian.z_pc,
        "source": o.source.reference,
    }
    for o in objects
]).sort_values("distance_pc").reset_index(drop=True)
df

,id,name,object_type,distance_pc,distance_error_pc,x_pc,y_pc,z_pc,source
0,sun,Sun,reference_point,0.000000,NaN,0.000000,0.000000,0.000000e+00,Origin of the heliocentric coordinate system b...
1,local-bubble-centroid,Local Bubble,bubble,35.114100,NaN,10.200000,33.600000,3.727593e-14,"Alves, M.I.R., Boulanger, F., Ferriere, K. & M..."
2,hyades-open-cluster,Hyades,star_cluster,47.500000,0.150000,-44.307124,0.287726,-1.711859e+01,"Gaia Collaboration, Babusiaux, C., van Leeuwen..."
3,scorpius-centaurus-association,Scorpius-Centaurus Association,stellar_association,134.300000,11.700000,125.569591,-19.022730,4.366811e+01,"de Zeeuw, P. T., Hoogerwerf, R., de Bruijne, J..."
4,pleiades-open-cluster,Pleiades,star_cluster,135.150000,0.430000,-120.392160,28.986432,-5.413905e+01,"Lodieu, N., Perez-Garrido, A., Smart, R. L., S..."
5,ophiuchus-rho-ophiuchi-molecular-cloud,Ophiuchus / Rho Ophiuchi Molecular Cloud,molecular_cloud,139.000000,6.900000,132.269790,-15.772158,3.971072e+01,"Position: SIMBAD, ""NAME Ophiuchus Molecular Cl..."
6,taurus-molecular-cloud,Taurus Molecular Cloud,molecular_cloud,147.000000,14.300000,-141.592411,16.132483,-3.606012e+01,"Position: SIMBAD, ""NAME Taurus Complex"" (ident..."
7,lupus-molecular-cloud,Lupus Molecular Cloud,molecular_cloud,151.000000,15.600000,135.304385,-51.938398,4.238073e+01,"Position: SIMBAD, ""Lupus 1"" (identification on..."
8,pipe-nebula,Pipe Nebula,molecular_cloud,163.000000,5.000000,162.315349,4.166881,1.433055e+01,"Dzib, S. A., Loinard, L., Ortiz-Leon, G. N., e..."
9,chamaeleon-molecular-cloud,Chamaeleon Molecular Cloud,molecular_cloud,210.000000,19.100000,93.530425,-179.670340,-5.541325e+01,"Position: SIMBAD, ""NAME Chamaeleon I"" / ""NAME ..."


## 2. Verify the coordinate transform

For every object, reconstructing the heliocentric distance from the derived
Cartesian XYZ (`sqrt(x² + y² + z²)`) must reproduce the originally stated
`distance.value_pc` — this is the same invariant `tests/test_coordinates.py`
and `tests/test_initial_catalog.py` check automatically; this cell re-derives
it visually/numerically as a validation step (spec §37).

In [3]:
reconstructed = np.sqrt(df["x_pc"] ** 2 + df["y_pc"] ** 2 + df["z_pc"] ** 2)
diff = (reconstructed - df["distance_pc"]).abs()
print(f"max |reconstructed - stated| distance error: {diff.max():.2e} pc")
assert diff.max() < 1e-3, "coordinate transform does not preserve distance"
print("OK: every object's derived XYZ reproduces its stated distance.")

max |reconstructed - stated| distance error: 3.41e-13 pc
OK: every object's derived XYZ reproduces its stated distance.


## 3. XYZ distribution of the initial object set

In [4]:
fig = px.scatter_3d(
    df,
    x="x_pc", y="y_pc", z="z_pc",
    color="object_type",
    hover_name="name",
    hover_data={"distance_pc": ":.1f", "source": True, "x_pc": False, "y_pc": False, "z_pc": False},
    title="Local Galactic neighborhood — initial catalog (heliocentric Galactic XYZ, pc)",
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(scene=dict(xaxis_title="X (pc, → Galactic Center)",
                              yaxis_title="Y (pc, → rotation)",
                              zaxis_title="Z (pc, → NGP)"),
                   height=700)
fig.show()

## 4. Compare against the scientific model layers

Overlay the Gould Belt (tilted ellipse/annulus), the Radcliffe Wave (fitted
spine polyline), and the Local Bubble (simplified ellipsoid, coarse wireframe —
its full Euler-angle orientation is not reproduced here, this is a diagnostic
approximation only) on the same axes as the catalog objects, so the two kinds
of large-scale interpretation can be visually compared against the same
physical dataset (spec Idea.md §49's acceptance criterion) without moving any
catalog object by hand.

In [5]:
def gould_belt_ellipse_points(model, n=200):
    t = np.linspace(0, 2 * np.pi, n)
    x0 = model.major_radius_pc * np.cos(t)
    y0 = model.minor_radius_pc * np.sin(t)
    incl = np.radians(model.inclination_deg)
    x1, y1, z1 = x0, y0 * np.cos(incl), y0 * np.sin(incl)
    orient = np.radians(model.orientation_deg)
    x2 = x1 * np.cos(orient) - y1 * np.sin(orient)
    y2 = x1 * np.sin(orient) + y1 * np.cos(orient)
    z2 = z1
    return (
        x2 + model.center.x_pc,
        y2 + model.center.y_pc,
        z2 + model.center.z_pc,
    )


def local_bubble_wireframe_points(model, n_theta=16, n_phi=24):
    theta = np.linspace(0, np.pi, n_theta)
    phi = np.linspace(0, 2 * np.pi, n_phi)
    theta, phi = np.meshgrid(theta, phi)
    a, b, c = model.semi_axes_pc.a_pc, model.semi_axes_pc.b_pc, model.semi_axes_pc.c_pc
    x = a * np.sin(theta) * np.cos(phi) + model.center_pc.x_pc
    y = b * np.sin(theta) * np.sin(phi) + model.center_pc.y_pc
    z = c * np.cos(theta) + model.center_pc.z_pc
    return x.ravel(), y.ravel(), z.ravel()


gx, gy, gz = gould_belt_ellipse_points(gould_belt)
lx, ly, lz = local_bubble_wireframe_points(local_bubble)
rx = [p.x_pc for p in radcliffe_wave.points]
ry = [p.y_pc for p in radcliffe_wave.points]
rz = [p.z_pc for p in radcliffe_wave.points]

fig2 = go.Figure()
fig2.add_trace(go.Scatter3d(
    x=df["x_pc"], y=df["y_pc"], z=df["z_pc"],
    mode="markers", marker=dict(size=4, color="lightgray"),
    text=df["name"], name="catalog objects",
))
fig2.add_trace(go.Scatter3d(
    x=gx, y=gy, z=gz, mode="lines",
    line=dict(color="orange", width=4), name=f"Gould Belt ({gould_belt.source.reference[:40]}…)",
))
fig2.add_trace(go.Scatter3d(
    x=rx, y=ry, z=rz, mode="lines",
    line=dict(color="deepskyblue", width=4), name="Radcliffe Wave (fitted spine)",
))
fig2.add_trace(go.Scatter3d(
    x=lx, y=ly, z=lz, mode="markers",
    marker=dict(size=1.5, color="mediumpurple", opacity=0.3), name="Local Bubble (coarse wireframe)",
))
fig2.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers",
    marker=dict(size=6, color="gold", symbol="diamond"), name="Sun (origin)",
))
fig2.update_layout(
    title="Catalog objects vs. Gould Belt / Radcliffe Wave / Local Bubble model layers",
    scene=dict(xaxis_title="X (pc)", yaxis_title="Y (pc)", zaxis_title="Z (pc)"),
    height=750,
)
fig2.show()

## 5. Diagnostic plots

In [6]:
fig3 = px.histogram(df, x="distance_pc", nbins=15, title="Distance distribution of the initial catalog (pc)")
fig3.show()

fig4 = px.scatter(
    df.dropna(subset=["distance_error_pc"]),
    x="distance_pc", y="distance_error_pc", color="object_type", hover_name="name",
    title="Distance uncertainty vs. distance (objects with a stated distance_error_pc)",
    labels={"distance_pc": "distance (pc)", "distance_error_pc": "distance error (pc)"},
)
fig4.show()